# 小恐龍自動遊玩機器人（長時間高分優化版）

在 Chrome 開啟 `chrome://dino` 後執行本 Notebook。

> 執行前請先點一下遊戲畫面，讓 Chrome 取得鍵盤焦點；過程中盡量不要移動視窗。

## 安全中止方式
- 按 **Esc**
- 將滑鼠移到螢幕**任一角落**（PyAutoGUI FailSafe）
- 在 Notebook 工具列按 **Interrupt（■）**

## 使用前校正
1. 先執行「校正輔助」cell，確認 `GROUND_Y`、`DINO_X`、`SCAN_X` 符合你的遊戲視窗。
2. 若遊戲視窗位置不同，請修改設定區的常數。
3. 執行主程式 cell，倒數 3 秒後開始。

## 優化重點
- 只截取小恐龍前方的小區域，不再抓全螢幕。
- 使用 NumPy 向量化判斷，避免逐像素 Python 迴圈。
- 依估計的遊戲速度動態加大提前起跳距離。
- 夜間模式自動反轉顏色判斷。
- 自動判斷仙人掌／翼龍，必要時低頭。
- 遊戲結束後會自動按空白鍵重新開始。
- `try...except...finally` 例外處理，結束時一定放開按鍵。


In [ ]:
import pyautogui as pyg
import time

# ============================================================
# 1) 匯入套件（開頭必備）
# ============================================================
try:
    import numpy as np
except ImportError as exc:
    raise ImportError(
        "缺少 numpy，請先在 iem_python 環境安裝：\n"
        "conda install -n iem_python -c conda-forge numpy pillow -y"
    ) from exc

# pyautogui 預設每次呼叫後會暫停 0.1 秒，關掉才跟得上高速遊戲
pyg.PAUSE = 0
pyg.FAILSAFE = True  # 滑鼠移到角落時觸發 FailSafeException

SCREEN_W, SCREEN_H = pyg.size()

# ============================================================
# 2) 偵測範圍設定（請依遊戲視窗位置微調）
# ============================================================
GROUND_Y = 641       # 障礙物偵測基準線 y（原程式使用 641）
DINO_X = 225         # 小恐龍左側 x，用來計算距離
SCAN_X = 239         # 偵測區左緣 x（原程式從 239 開始）
SCAN_W = 520         # 偵測區寬度；越大越早看到障礙物

SCAN_TOP = GROUND_Y - 65          # 往上抓 65 px，足以涵蓋仙人掌與中空翼龍
SCAN_BOTTOM = GROUND_Y - 2        # 停在實際地面線上方
SCAN_H = SCAN_BOTTOM - SCAN_TOP + 1

# 取樣列（相對 GROUND_Y 的高度；負值代表上方）
LOW_ROWS = [-4, -10, -18]   # 仙人掌、低空翼龍 → 要跳
MID_ROWS = [-32, -40, -48]  # 中空翼龍 → 要低頭

# 顏色門檻（日間：深色是障礙物；夜間：亮色是障礙物）
DARK_LEVEL = 130
LIGHT_LEVEL = 150

# 動態提前量
LEAD_TIME = 0.30       # 依「時間」提前，速度越快距離越大
LEAD_MARGIN = 12       # 額外安全距離（px）
JUMP_COOLDOWN = 0.25   # 兩次跳躍最短間隔（秒）
DUCK_HOLD = 0.35       # 翼龍通過後繼續低頭的時間（秒）
DEFAULT_SPEED = 360.0  # 尚未測到速度時的預設值（px/秒）
RESTART_AFTER = 4.0    # 超過幾秒沒障礙物就按空白鍵開始／重來
STUCK_AFTER = 1.0      # 障礙物完全不動超過幾秒，視為遊戲結束並重來
ENABLE_BIRD = True     # 是否自動低頭閃中空翼龍

# 將相對列轉成截圖內的索引
YS_LOW = [GROUND_Y + dy - SCAN_TOP for dy in LOW_ROWS]
YS_MID = [GROUND_Y + dy - SCAN_TOP for dy in MID_ROWS]

print(f"螢幕尺寸：{SCREEN_W} x {SCREEN_H}")
print(f"偵測區域：x={SCAN_X}..{SCAN_X + SCAN_W - 1}, y={SCAN_TOP}..{SCAN_BOTTOM}")
print("安全中止：按 Esc，或把滑鼠移到螢幕任一角落。")


In [ ]:
# ============================================================
# 3) 安全防呆：Esc／滑鼠角落／例外處理
# ============================================================
try:
    import ctypes
    _user32 = ctypes.windll.user32
except Exception:
    _user32 = None

_VK_ESCAPE = 0x1B


def esc_pressed():
    """Windows 下直接偵測 Esc 是否被按住，不需額外套件。"""
    if _user32 is None:
        return False
    return bool(_user32.GetAsyncKeyState(_VK_ESCAPE) & 0x8000)


def mouse_in_corner(margin=3):
    """滑鼠移到任一角落時回傳 True。"""
    x, y = pyg.position()
    return (
        x <= margin or y <= margin
        or x >= SCREEN_W - 1 - margin or y >= SCREEN_H - 1 - margin
    )


# ============================================================
# 4) 影像偵測與速度估計
# ============================================================
def grab_band():
    """只截取小恐龍前方的一小條區域，比全螢幕截圖快很多。"""
    return pyg.screenshot(region=(SCAN_X, SCAN_TOP, SCAN_W, SCAN_H))


def build_mask(im):
    """回傳 (night, mask)；mask 為 True 的像素代表障礙物。"""
    arr = np.asarray(im)
    if arr.ndim == 2:
        gray = arr.astype(float)
    else:
        gray = arr[:, :, :3].mean(axis=2)
    night = float(gray.mean()) < 100.0
    mask = gray > LIGHT_LEVEL if night else gray < DARK_LEVEL
    return night, mask


def nearest_x(cols):
    """回傳最左邊 True 的螢幕 x；沒有則回傳 None。"""
    idx = np.flatnonzero(cols)
    if idx.size == 0:
        return None
    return SCAN_X + int(idx[0])


class SpeedEstimator:
    """用障礙物在畫面上的位移估計遊戲速度（px/秒）。"""

    def __init__(self, initial=DEFAULT_SPEED, alpha=0.35):
        self.speed = float(initial)
        self.alpha = alpha
        self.last_x = None
        self.last_t = None

    def update(self, x, now):
        # 太靠近小恐龍時，障礙物邊緣會被裁切，停止更新
        if x is None or x < DINO_X + 25:
            self.last_x = None
            return
        if self.last_x is not None and self.last_t is not None:
            dx = self.last_x - x      # 障礙物向左移動，正值代表前進
            dt = now - self.last_t
            if dt > 1e-4 and 0 < dx < 200:
                raw = dx / dt
                if 100.0 <= raw <= 2000.0:
                    self.speed = (1 - self.alpha) * self.speed + self.alpha * raw
        self.last_x = x
        self.last_t = now

    def lead(self):
        """速度越快，提前起跳的距離越大。"""
        return min(SCAN_W - 30.0, max(60.0, self.speed * LEAD_TIME + LEAD_MARGIN))


class DinoController:
    """集中管理跳躍與低頭，確保按鍵一定會被放開。"""

    def __init__(self):
        self.ducking = False
        self.last_jump = 0.0

    def jump(self, now):
        if now - self.last_jump < JUMP_COOLDOWN:
            return False
        self.stand()
        pyg.press('space')
        self.last_jump = now
        return True

    def duck(self):
        if not self.ducking:
            pyg.keyDown('down')
            self.ducking = True

    def stand(self):
        if self.ducking:
            pyg.keyUp('down')
            self.ducking = False

    def release(self):
        self.stand()


In [ ]:
# ============================================================
# 5) 校正輔助（原本壞掉的 cell 已修正）
# ============================================================
try:
    pyg.sleep(1)
    im = pyg.screenshot()
    print("螢幕尺寸：", im.size)
    print(f"目前設定：GROUND_Y={GROUND_Y}, DINO_X={DINO_X}, SCAN_X={SCAN_X}, SCAN_W={SCAN_W}")
    print(f"偵測區：({SCAN_X}, {SCAN_TOP}) 到 ({SCAN_X + SCAN_W - 1}, {SCAN_BOTTOM})")

    for name, (x, y) in {
        "地面基準點": (SCAN_X, GROUND_Y),
        "仙人掌取樣列": (SCAN_X + 60, GROUND_Y - 10),
        "中空翼龍取樣列": (SCAN_X + 60, GROUND_Y - 40),
    }.items():
        print(f"{name} {(x, y)} 的顏色：{im.getpixel((x, y))}")

    print("提示：白色 (247,247,247) 代表該位置目前沒有障礙物，屬正常現象。")
    print("若顏色與實際畫面不符，請調整上方設定區的座標。")
except Exception as exc:
    print(f"校正失敗：{exc!r}")
    print("請確認 pyautogui 可以正常截圖（需要 Pillow）。")


In [ ]:
# ============================================================
# 6) 主程式：長時間自動遊玩
# ============================================================
def play_dino():
    print("=" * 60)
    print("小恐龍自動遊玩啟動")
    print("安全中止：按 Esc、滑鼠移到螢幕角落，或按 Notebook 的 Interrupt。")
    print("=" * 60)

    ctrl = DinoController()
    speed = SpeedEstimator()
    last_obstacle_t = time.perf_counter()
    last_bird_t = -DUCK_HOLD
    last_obstacle_x = None
    last_move_t = time.perf_counter()
    frames = 0
    jumps = 0
    started = time.perf_counter()

    try:
        for i in range(3, 0, -1):
            if esc_pressed() or mouse_in_corner():
                print("\n已取消啟動。")
                return
            print(f"{i} 秒後開始...")
            time.sleep(1)

        pyg.press('space')  # 開始遊戲；若已在遊戲中只會跳一下
        while True:
            now = time.perf_counter()

            if esc_pressed() or mouse_in_corner():
                print("\n偵測到 Esc 或滑鼠在角落，安全中止。")
                break

            im = grab_band()
            night, mask = build_mask(im)

            low_cols = mask[YS_LOW, :].any(axis=0)
            mid_cols = mask[YS_MID, :].any(axis=0)

            cactus_x = nearest_x(low_cols)
            bird_x = nearest_x(mid_cols & ~low_cols)  # 不碰地才算翼龍

            speed.update(cactus_x, now)
            lead = speed.lead()

            # 追蹤最近的障礙物是否還在移動（用來判斷遊戲是否已結束）
            current_x = cactus_x if cactus_x is not None else bird_x
            if current_x is not None:
                if last_obstacle_x is not None and abs(current_x - last_obstacle_x) > 2:
                    last_move_t = now
                last_obstacle_x = current_x
            else:
                last_obstacle_x = None
                last_move_t = now

            cactus_threat = (
                cactus_x is not None
                and 0 <= cactus_x - DINO_X <= lead
            )
            bird_threat = (
                ENABLE_BIRD
                and bird_x is not None
                and 0 <= bird_x - DINO_X <= lead + 20
            )

            if cactus_x is not None or bird_x is not None:
                last_obstacle_t = now
            if bird_threat:
                last_bird_t = now

            if cactus_threat:
                if ctrl.jump(now):
                    jumps += 1
            elif bird_threat or (now - last_bird_t < DUCK_HOLD):
                ctrl.duck()
            else:
                ctrl.stand()

            # 太久沒看到障礙物，或障礙物完全不動 → 可能還沒開始或已經結束
            game_stuck = (
                current_x is not None
                and now - last_move_t > STUCK_AFTER
            )
            if now - last_obstacle_t > RESTART_AFTER or game_stuck:
                ctrl.stand()
                pyg.press('space')
                last_obstacle_t = now
                last_move_t = now
                last_obstacle_x = None

            frames += 1

    except pyg.FailSafeException:
        print("\n滑鼠移到螢幕角落，已觸發 FailSafe，安全中止。")
    except KeyboardInterrupt:
        print("\n使用者中斷，已安全中止。")
    except pyg.PyAutoGUIException as exc:
        print("\nPyAutoGUI 發生錯誤，通常是缺少 Pillow 或截圖失敗：")
        print(f"  {exc}")
        print("請在 iem_python 環境安裝：")
        print("  conda install -n iem_python -c conda-forge pillow numpy -y")
    except Exception as exc:
        print(f"\n發生未預期錯誤：{exc!r}")
        print("已安全停止，請檢查上方訊息。")
    finally:
        try:
            ctrl.release()
        except Exception:
            pass
        elapsed = time.perf_counter() - started
        fps = frames / elapsed if elapsed > 0 else 0.0
        print("-" * 60)
        print(f"執行時間：{elapsed:.1f} 秒")
        print(f"處理影格：{frames}（平均 {fps:.1f} FPS）")
        print(f"跳躍次數：{jumps}")
        print(f"估計遊戲速度：{speed.speed:.0f} px/秒")
        print(f"目前提前量：{speed.lead():.0f} px")
        print("已放開所有按鍵，程式結束。")


play_dino()
